# SAM2: Segmentacion Universal de Imagenes

SAM (Segment Anything Model) y su version mejorada SAM2 son modelos que pueden segmentar cualquier objeto en una imagen sin entrenamiento previo.

**Que hace SAM2:**
- Segmenta objetos a nivel de pixel (mascaras precisas)
- Funciona con puntos, boxes o automaticamente
- No necesita saber que objeto es, solo lo segmenta
- Se puede combinar con detectores (Grounding DINO, Qwen) para segmentacion guiada por texto

En este notebook veremos SAM2 solo y combinado con otros modelos para crear pipelines potentes.

## Configuracion e Imports

In [ ]:
try:
    import qwen_vl_utils
except ImportError:
    %pip install qwen-vl-utils
    import qwen_vl_utils

In [ ]:
import torch
import transformers
from transformers import SamProcessor, SamModel
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from transformers import Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
import requests
import json
import re
from io import BytesIO
import warnings
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version: {torch.__version__}, usando: {device}")
print(f"transformers version: {transformers.__version__}")

## Carga de Modelos

Cargamos SAM2 y los modelos que combinaremos con el: Grounding DINO y Qwen2.5-VL.

**Nota**: SAM2 usa `sam2-hiera-large`. Si tienes poca memoria GPU, puedes usar `sam2-hiera-base-plus`.

In [ ]:
# SAM2
sam_name = "facebook/sam-hiera-large"
sam_processor = SamProcessor.from_pretrained(sam_name)
sam_model = SamModel.from_pretrained(sam_name).to(device)
print(f"✓ SAM2 cargado")

# Grounding DINO
dino_name = "IDEA-Research/grounding-dino-base"
dino_processor = AutoProcessor.from_pretrained(dino_name)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(dino_name).to(device)
print(f"✓ Grounding DINO cargado")

# Qwen2.5-VL
qwen_name = "Qwen/Qwen2.5-VL-3B-Instruct"
qwen_processor = AutoProcessor.from_pretrained(qwen_name)
qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(qwen_name, torch_dtype=torch.float16, device_map="auto")
print(f"✓ Qwen2.5-VL cargado")

print("\nTodos los modelos listos!")

## Carga de Imagenes de Ejemplo

In [ ]:
url_person_cars = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/person_cars.jpg"
image_person_cars = Image.open(BytesIO(requests.get(url_person_cars).content)).convert("RGB")

url_fruits = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/fruits.jpg"
image_fruits = Image.open(BytesIO(requests.get(url_fruits).content)).convert("RGB")

url_bananas = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/bananas.jpg"
image_bananas = Image.open(BytesIO(requests.get(url_bananas).content)).convert("RGB")

print("Imagenes cargadas")

## SAM2: Segmentacion con Bounding Box

La forma mas simple de usar SAM2 es darle una bounding box. SAM2 genera una mascara precisa del objeto dentro de esa caja.

**Formato de box**: `[x_min, y_min, x_max, y_max]`

In [ ]:
def segment_from_box(image, box):
    """Segmenta objeto usando SAM2 con una bounding box"""
    
    inputs = sam_processor(images=image, input_boxes=[[box]], return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = sam_model(**inputs)
    
    # Post-procesar mascaras
    masks = sam_processor.post_process_masks(
        outputs.pred_masks,
        inputs["original_sizes"],
        inputs["reshaped_input_sizes"]
    )
    
    # SAM2 genera 3 mascaras, elegimos la mejor segun IoU score
    best_idx = outputs.iou_scores.argmax().item()
    best_mask = masks[0][0][best_idx].cpu().numpy()
    
    # Visualizar
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    axes[1].imshow(best_mask, cmap="gray")
    axes[1].set_title("Mascara")
    axes[1].axis("off")
    
    masked_img = np.array(image) * best_mask[:, :, None]
    axes[2].imshow(masked_img.astype(np.uint8))
    axes[2].set_title("Resultado")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    return best_mask

# Ejemplo: segmentar la persona (ajusta las coordenadas segun la imagen)
box_person = [290, 90, 440, 450]  # [x_min, y_min, x_max, y_max]
mask = segment_from_box(image_person_cars, box_person)

## Pipeline: Grounding DINO + SAM2

Ahora combinamos Grounding DINO (deteccion por texto) con SAM2 (segmentacion precisa):

1. Grounding DINO detecta objetos usando un prompt de texto → genera boxes
2. SAM2 toma esas boxes y genera mascaras precisas
3. Resultado: segmentacion guiada por texto!

Este pipeline es muy util en industria: "segmenta todas las tuercas", "segmenta los componentes defectuosos", etc.

In [ ]:
def segment_with_text(image, prompt, box_threshold=0.35, text_threshold=0.25):
    """Pipeline: Grounding DINO + SAM2 para segmentar con texto"""
    
    w, h = image.size
    
    # Paso 1: Detectar con Grounding DINO
    inputs_dino = dino_processor(images=image, text=prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs_dino = dino_model(**inputs_dino)
    
    results = dino_processor.post_process_grounded_object_detection(
        outputs_dino,
        inputs_dino.input_ids,
        threshold=box_threshold,
        text_threshold=text_threshold,
        target_sizes=[(h, w)]
    )
    
    boxes = results[0]['boxes'].cpu().numpy().tolist()
    labels = results[0]['labels']
    
    print(f"Detectados {len(boxes)} objetos: {labels}")
    
    if len(boxes) == 0:
        print("No se detectaron objetos. Intenta bajar los thresholds.")
        return None
    
    # Paso 2: Segmentar con SAM2 (batch mode para todas las boxes)
    inputs_sam = sam_processor(images=image, input_boxes=[boxes], return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs_sam = sam_model(**inputs_sam)
    
    masks = sam_processor.post_process_masks(
        outputs_sam.pred_masks,
        inputs_sam["original_sizes"],
        inputs_sam["reshaped_input_sizes"]
    )
    
    iou_scores = outputs_sam.iou_scores.cpu().numpy()
    
    # Combinar todas las mascaras
    final_mask = np.zeros((h, w), dtype=np.float32)
    
    for i in range(len(boxes)):
        detection_masks = masks[0][i].cpu().numpy()
        best_idx = iou_scores[0][i].argmax()
        final_mask += detection_masks[best_idx]
    
    final_mask = np.clip(final_mask, 0, 1)
    
    # Visualizar
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    
    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    axes[1].imshow(final_mask, cmap="gray")
    axes[1].set_title("Mascara")
    axes[1].axis("off")
    
    masked_img = np.array(image) * final_mask[:, :, None]
    axes[2].imshow(masked_img.astype(np.uint8))
    axes[2].set_title(f"Resultado: {prompt}")
    axes[2].axis("off")
    
    inverted_mask = 1 - final_mask
    removed_img = np.array(image) * inverted_mask[:, :, None]
    axes[3].imshow(removed_img.astype(np.uint8))
    axes[3].set_title("Fondo sin objetos")
    axes[3].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    return final_mask

# Ejemplo: segmentar frutas especificas
mask_fruits = segment_with_text(image_fruits, "kiwi. apple.")

## Pipeline: Qwen2.5-VL + SAM2

Este es el pipeline mas avanzado. Usa Qwen2.5-VL para:
- Entender instrucciones complejas ("segmenta las frutas que no son de Valencia")
- Razonar sobre la imagen
- Generar bounding boxes con JSON

Luego SAM2 convierte esas boxes en mascaras precisas.

In [ ]:
def segment_with_qwen(image, instruction):
    """Pipeline: Qwen2.5-VL + SAM2 para segmentar con razonamiento"""
    
    w, h = image.size
    
    # Paso 1: Detectar con Qwen2.5-VL
    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": instruction}
        ]}
    ]
    
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs_qwen = qwen_processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(device)
    
    with torch.no_grad():
        generated_ids = qwen_model.generate(**inputs_qwen, max_new_tokens=256)
    
    answer = qwen_processor.batch_decode(
        [out[len(inp):] for inp, out in zip(inputs_qwen.input_ids, generated_ids)],
        skip_special_tokens=True
    )[0]
    
    print(f"Qwen responde: {answer}\n")
    
    # Extraer boxes del JSON
    json_match = re.search(r'```json\n(.*?)\n```', answer, re.DOTALL)
    
    if not json_match:
        print("No se encontraron boxes en la respuesta")
        return None
    
    data = json.loads(json_match.group(1))
    boxes = []
    labels = []
    
    for item in data:
        if 'bbox_2d' in item:
            boxes.append(item['bbox_2d'])
            labels.append(item.get('label', 'object'))
    
    print(f"Detectados {len(boxes)} objetos: {labels}")
    
    if len(boxes) == 0:
        print("No hay boxes para segmentar")
        return None
    
    # Paso 2: Segmentar con SAM2
    inputs_sam = sam_processor(images=image, input_boxes=[boxes], return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs_sam = sam_model(**inputs_sam)
    
    masks = sam_processor.post_process_masks(
        outputs_sam.pred_masks,
        inputs_sam["original_sizes"],
        inputs_sam["reshaped_input_sizes"]
    )
    
    iou_scores = outputs_sam.iou_scores.cpu().numpy()
    
    # Combinar mascaras
    final_mask = np.zeros((h, w), dtype=np.float32)
    
    for i in range(len(boxes)):
        detection_masks = masks[0][i].cpu().numpy()
        best_idx = iou_scores[0][i].argmax()
        final_mask += detection_masks[best_idx]
    
    final_mask = np.clip(final_mask, 0, 1)
    
    # Aplicar filtros morfologicos para limpiar ruido
    final_mask = cv2.morphologyEx(final_mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
    final_mask = cv2.morphologyEx(final_mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
    
    # Visualizar
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    
    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    axes[1].imshow(final_mask, cmap="gray")
    axes[1].set_title("Mascara")
    axes[1].axis("off")
    
    masked_img = np.array(image) * final_mask[:, :, None]
    axes[2].imshow(masked_img.astype(np.uint8))
    axes[2].set_title("Resultado")
    axes[2].axis("off")
    
    inverted_mask = 1 - final_mask
    removed_img = np.array(image) * inverted_mask[:, :, None]
    axes[3].imshow(removed_img.astype(np.uint8))
    axes[3].set_title("Fondo")
    axes[3].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    return final_mask

# Ejemplo: instruccion compleja
mask_qwen = segment_with_qwen(
    image_fruits,
    "Detect the fruits that are from Valencia (oranges) and provide bounding box coordinates."
)

## Caso Practico: Control de Calidad

Vamos a simular un sistema de control de calidad para clasificar bananas por tamaño. Segmentamos todas las bananas y medimos su area.

In [ ]:
def quality_control(image, prompt, area_threshold=9000):
    """Control de calidad: segmenta y clasifica por tamaño"""
    
    # Segmentar objetos
    mask = segment_with_text(image, prompt, box_threshold=0.3, text_threshold=0.2)
    
    if mask is None:
        return
    
    # Encontrar contornos
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    img_result = np.array(image)
    ok_count = 0
    nok_count = 0
    
    for contour in contours:
        area = cv2.contourArea(contour)
        
        if area > 100:  # Filtrar contornos pequeños
            is_ok = area >= area_threshold
            color = (0, 255, 0) if is_ok else (255, 0, 0)  # Verde=OK, Rojo=NOK
            
            if is_ok:
                ok_count += 1
            else:
                nok_count += 1
            
            # Dibujar contorno
            cv2.drawContours(img_result, [contour], -1, color, 3)
            
            # Añadir texto con area
            M = cv2.moments(contour)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                lowest_y = max(contour[:, 0, 1])
                cv2.putText(img_result, f"Area: {int(area)}", (cx-60, lowest_y+20),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    # Visualizar
    plt.figure(figsize=(10, 6))
    plt.imshow(img_result)
    plt.title(f"Control de Calidad - OK: {ok_count} (verde) | NOK: {nok_count} (rojo)")
    plt.axis("off")
    plt.show()
    
    print(f"\nResultados:")
    print(f"  ✓ Objetos OK (>= {area_threshold} px): {ok_count}")
    print(f"  ✗ Objetos NOK (< {area_threshold} px): {nok_count}")

# Aplicar control de calidad a bananas
quality_control(image_bananas, "banana", area_threshold=9000)

## Ejercicio: Segmenta tus propios objetos

Prueba los pipelines con diferentes imagenes y prompts:

1. Usa `segment_with_text()` para segmentar los coches en `image_person_cars`
2. Usa `segment_with_qwen()` con una instruccion creativa
3. Experimenta con los thresholds para mejorar las detecciones

In [ ]:
# EJERCICIO: Tu codigo aqui

# Ejemplo 1: Segmentar coches
# mask_cars = segment_with_text(image_person_cars, "car.")

# Ejemplo 2: Instruccion creativa con Qwen
# mask_creative = segment_with_qwen(image_fruits, "Tu instruccion aqui...")


## Ejercicio Extra (Comodin): Calcula perimetros

Modifica la funcion `quality_control()` para que ademas del area, calcule y muestre el perimetro de cada objeto.

**Pista**: Usa `cv2.arcLength(contour, True)` para calcular el perimetro.

In [ ]:
# EJERCICIO EXTRA: Calcula perimetros
def quality_control_advanced(image, prompt, area_threshold=9000, perimeter_threshold=500):
    """Control de calidad con area y perimetro"""
    # Tu codigo aqui...
    pass

# Prueba:
# quality_control_advanced(image_bananas, "banana")


## Resumen

En este notebook hemos aprendido:

✅ Que es SAM2 y como funciona la segmentacion universal  
✅ Segmentar objetos con bounding boxes  
✅ Pipeline Grounding DINO + SAM2 (deteccion + segmentacion por texto)  
✅ Pipeline Qwen2.5-VL + SAM2 (razonamiento + segmentacion)  
✅ Aplicacion practica: control de calidad industrial  
✅ Procesamiento de mascaras y analisis de contornos  

**Siguiente paso**: En el proximo notebook veremos estimacion de pose humana para detectar keypoints del cuerpo.

**Contacto**: Si tienes dudas puedes escribirme un email!